# 03 · Join Sofascore + Capology — Turkey Süper Lig 24/25

Integración de estadísticas de rendimiento (Sofascore) con datos salariales (Capology)
para la temporada **2024/25 de Süper Lig turca**.

**Flujo de matching:**
1. Normalización de nombres (tildes, mayúsculas, caracteres especiales)
2. `TEAM_MAP`: alineación manual de nombres de equipo entre fuentes
3. Merge exacto normalizado
4. Fuzzy matching en cuatro niveles:
   - Score ≥ 0.90 → aceptación automática
   - 0.75 ≤ score < 0.90 → revisión manual
   - 0.50 ≤ score < 0.75 → revisión manual estricta
   - score < 0.50 → revisión manual muy estricta
5. Revisión de jugadores sin salario
6. Guardado en `data/master/`

---

## 1. Imports y rutas

In [1]:
import pandas as pd
import unicodedata
import re
from rapidfuzz import fuzz
from pathlib import Path
from IPython.display import display

ROOT       = Path.cwd().parents[1]
SF_DIR     = ROOT / 'data' / 'processed' / 'sofascore'
CG_DIR     = ROOT / 'data' / 'processed' / 'capology'
MASTER_DIR = ROOT / 'data' / 'master'
MASTER_DIR.mkdir(parents=True, exist_ok=True)

print('✅ Rutas configuradas')
print(f'   Root:   {ROOT}')
print(f'   Master: {MASTER_DIR}')

✅ Rutas configuradas
   Root:   d:\USER\Desktop\TFM
   Master: d:\USER\Desktop\TFM\data\master


## 2. Función de normalización

In [2]:
def normalize(s):
    """
    Normaliza un string para comparación: elimina tildes, pasa a minúsculas,
    elimina caracteres especiales y espacios extra.
    """
    if pd.isna(s):
        return ''
    s = str(s)
    s = unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode('ascii')
    s = re.sub(r'[^a-z0-9\s]', ' ', s.lower().strip())
    return re.sub(r'\s+', ' ', s).strip()

print('✅ Función definida')

✅ Función definida


## 3. Carga de datos

In [3]:
df_sf = pd.read_csv(SF_DIR / 'df_turkey_2425.csv').copy()
df_cg = pd.read_csv(CG_DIR / 'cg_turkey_2425.csv').copy()

print(f'Sofascore:  {df_sf.shape[0]} jugadores | {df_sf.shape[1]} columnas')
print(f'Capology:   {df_cg.shape[0]} jugadores | {df_cg.shape[1]} columnas')

Sofascore:  580 jugadores | 116 columnas
Capology:   637 jugadores | 9 columnas


## 4. Normalización

In [4]:
df_sf['player_norm'] = df_sf['player'].apply(normalize)
df_sf['team_norm']   = df_sf['team'].apply(normalize)
df_cg['player_norm'] = df_cg['player'].apply(normalize)
df_cg['team_norm']   = df_cg['club'].apply(normalize)

print('✅ Normalización aplicada')

✅ Normalización aplicada


## 5. Alineación de equipos (TEAM_MAP)

### 5.1 Identificar discrepancias de nombres de equipo

In [5]:
solo_sf = set(df_sf['team_norm'].unique()) - set(df_cg['team_norm'].unique())
solo_cg = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())

print('En Sofascore pero no en Capology:')
for e in sorted(solo_sf): print(f'   {e}')
print()
print('En Capology pero no en Sofascore:')
for e in sorted(solo_cg): print(f'   {e}')

En Sofascore pero no en Capology:
   basaksehir fk
   besiktas jk
   bodrum fk
   gaziantep fk
   kasmpasa

En Capology pero no en Sofascore:
   basaksehir
   besiktas
   bodrum
   gaziantep bb
   kasimpasa


### 5.2 Aplicar TEAM_MAP

Rellenar con las discrepancias identificadas en la celda anterior.

In [7]:
# ── Ajustar según la celda anterior ──────────────────────────
TEAM_MAP = {'basaksehir':'basaksehir fk',
            'besiktas':'besiktas jk',
            'bodrum':'bodrum fk',
            'gaziantep bb':'gaziantep fk',
            'kasimpasa':'kasmpasa'
}
# ─────────────────────────────────────────────────────────────

df_cg['team_norm'] = df_cg['team_norm'].replace(TEAM_MAP)

diff = set(df_cg['team_norm'].unique()) - set(df_sf['team_norm'].unique())
if diff:
    print(f'⚠️  Equipos de CG aún sin match en SF: {diff}')
else:
    print('✅ Todos los equipos alineados')


✅ Todos los equipos alineados


## 6. Merge exacto normalizado

In [8]:
df_merged = df_sf.merge(
    df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
            'position', 'age', 'nationality']],
    on=['player_norm', 'team_norm'],
    how='left'
)

matched = df_merged['gross_annual_eur'].notna().sum()
total   = len(df_merged)

print(f'Merge exacto: {matched}/{total} ({matched/total:.1%})')
print(f'Sin emparejar: {total - matched}')

Merge exacto: 479/580 (82.6%)
Sin emparejar: 101


## 7. Fuzzy matching sobre los no emparejados

Se generan candidatos para todos los jugadores sin match exacto,
sin umbral mínimo, y se clasifican en cuatro niveles.

In [9]:
df_unmatched = df_merged[df_merged['gross_annual_eur'].isna()].copy()
cg_by_team   = df_cg.groupby('team_norm')['player_norm'].apply(list).to_dict()

rows = []
for _, row in df_unmatched[['player','team','player_norm','team_norm']].drop_duplicates().iterrows():
    candidates = cg_by_team.get(row['team_norm'], [])
    best_match, best_score = None, 0
    for cand in candidates:
        score = fuzz.ratio(row['player_norm'], cand) / 100
        if score > best_score:
            best_score = score
            best_match = cand
    if best_match is not None:
        rows.append({
            'player_sf'  : row['player'],
            'team'       : row['team'],
            'player_norm': row['player_norm'],
            'team_norm'  : row['team_norm'],
            'cg_match'   : best_match,
            'score'      : round(best_score, 3)
        })

df_candidates = pd.DataFrame(rows).sort_values('score', ascending=False)
auto_matches     = df_candidates[df_candidates['score'] >= 0.90].copy()
review_matches   = df_candidates[(df_candidates['score'] >= 0.75) & (df_candidates['score'] < 0.90)].copy()
low_matches      = df_candidates[(df_candidates['score'] >= 0.50) & (df_candidates['score'] < 0.75)].copy()
very_low_matches = df_candidates[df_candidates['score'] < 0.50].copy()

print(f'Auto-aceptados    (score ≥ 0.90):          {len(auto_matches)}')
print(f'Revisión media    (0.75 ≤ score < 0.90):   {len(review_matches)}')
print(f'Revisión estricta (0.50 ≤ score < 0.75):   {len(low_matches)}')
print(f'Revisión muy est. (score < 0.50):           {len(very_low_matches)}')

Auto-aceptados    (score ≥ 0.90):          15
Revisión media    (0.75 ≤ score < 0.90):   24
Revisión estricta (0.50 ≤ score < 0.75):   38
Revisión muy est. (score < 0.50):           24


### 7.1 Matches automáticos (score ≥ 0.90)

Revisar para confirmar que todos son correctos.

In [10]:
auto_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
21,Przemysław Frankowski,Galatasaray,przemyslaw frankowski,0.976
46,Dimitris Kolovetsios,Kayserispor,dimitrios kolovetsios,0.976
47,Bakhtiyar Zaynutdinov,Beşiktaş JK,baktiyar zaynutdinov,0.976
11,Husniddin Alikulov,Çaykur Rizespor,khusniddin alikulov,0.973
53,Christophe Lungoyi,Gaziantep FK,christopher lungoyi,0.973
42,Talha Sariarslan,Kayserispor,talha sararslan,0.968
12,Kacper Kozłowski,Gaziantep FK,kacper kozlowski,0.968
10,Jakub Kałuziński,Antalyaspor,jakub kaluzinski,0.968
52,Celal Dumanli,Bodrum FK,celal dumanl,0.960
15,Bilal Bayazit,Kayserispor,bilal bayazt,0.960


### 7.2 Revisión media (0.75 ≤ score < 0.90)

Añadir a `EXCLUDE_FROM_FUZZY` el `player_norm` de los incorrectos.

In [11]:
review_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
66,Muhammet Ali Özbaskıcı,Samsunspor,muhammet ozbaskc,0.889
71,Poyraz Efe Yıldırım,Trabzonspor,poyraz yldrm,0.857
20,Deniz Eren Dönmezer,Adana Demirspor,deniz donmezer,0.848
33,Đorđe Nikolić,Sivasspor,djordje nikolic,0.846
58,Baran Ali Gezek,Kayserispor,baran gezek,0.846
64,Godfrey Bitok Stephen,Gaziantep FK,godfrey stephen,0.833
89,Osman Ertuğrul Çetin,Fenerbahçe,ertugrul cetin,0.824
82,Taylan Utku Aydın,Kasımpaşa,taylan aydn,0.815
91,İzzet Furkan Malak,Göztepe,furkan malak,0.800
68,Tayyip Talha Sanuç,Beşiktaş JK,tayyip sanuc,0.800


In [12]:
# ── Falsos positivos a excluir del nivel medio ────────────────
EXCLUDE_FROM_FUZZY = ['yusuf karademir'
]
# ─────────────────────────────────────────────────────────────

review_accepted = review_matches[~review_matches['player_norm'].isin(EXCLUDE_FROM_FUZZY)]
print(f'Aceptados: {len(review_accepted)} | Excluidos: {len(EXCLUDE_FROM_FUZZY)}')


Aceptados: 23 | Excluidos: 1


### 7.3 Revisión estricta (0.50 ≤ score < 0.75)

Por defecto ninguno se acepta. Añadir a `ACCEPT_LOW_FUZZY` los correctos.

In [13]:
low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
17,Eren Elmalı,Galatasaray,evren eren elmali,0.741
65,Mehmet Eray Özbek,Kayserispor,eray ozbek,0.741
37,Muhammed Birkan Tetik,Eyüpspor,birkan tetik,0.727
30,Muhammet Tunahan Taşçı,Konyaspor,tunahan tasc,0.727
90,Yakup Arda Kılıç,Beşiktaş JK,arda klc,0.727
5,Muhammed-Cham Saračević,Trabzonspor,muhammed cham,0.722
23,Thalisson Kelven,Antalyaspor,thalisson,0.720
3,Barış Alper Yılmaz,Galatasaray,baris yilmaz,0.714
43,Boran Başkan,Trabzonspor,serkan asan,0.696
16,Mutassim Al-Musrati,Beşiktaş JK,al musrati,0.690


In [20]:
# ── Matches de score bajo confirmados manualmente ─────────────
ACCEPT_LOW_FUZZY = ['eren elmal',
                    'mehmet eray ozbek',
                    'muhammed birkan tetik',
                    'muhammet tunahan tasc',
                    'yakup arda klc',
                    'muhammed cham saracevic',
                    'thalisson kelven',
                    'bars alper ylmaz',
                    'mutassim al musrati',
                    'muhammed sinan kaya',
                    'djalma',
                    'seyfettin anl yasar',
                    'anderson talisca'
                    
                    
                    

]
# ─────────────────────────────────────────────────────────────

low_accepted = low_matches[low_matches['player_norm'].isin(ACCEPT_LOW_FUZZY)]
print(f'Aceptados del nivel bajo: {len(low_accepted)}')


Aceptados del nivel bajo: 13


### 7.4 Revisión muy estricta (score < 0.50)

Por defecto ninguno se acepta. Añadir a `ACCEPT_VERY_LOW_FUZZY` los correctos.

In [21]:
very_low_matches[['player_sf', 'team', 'cg_match', 'score']]

,player_sf,team,cg_match,score
28,Marius Mouandilmadji,Samsunspor,marc bola,0.483
61,Sokol Cikalleshi,Konyaspor,josip calusic,0.483
99,Ismail Zobu,Hatayspor,bilal boutobba,0.480
50,José Rodríguez,Adana Demirspor,semih guler,0.480
27,Burak Çoban,Bodrum FK,gabriel obekpa,0.480
2,Svit Sešlar,Eyüpspor,huseyin maldar,0.480
18,Omar Colley,Beşiktaş JK,joao mario,0.476
34,Yusuf Inci,Kasımpaşa,yasin ozcan,0.476
57,Yigit Ali Buz,Hatayspor,ali yldz,0.476
81,Deniz Aksoy,Hatayspor,emir daduk,0.476


In [22]:
# ── Matches very low confirmados manualmente ──────────────────
ACCEPT_VERY_LOW_FUZZY = [

]
# ─────────────────────────────────────────────────────────────

very_low_accepted = very_low_matches[very_low_matches['player_norm'].isin(ACCEPT_VERY_LOW_FUZZY)]
print(f'Aceptados del nivel very low: {len(very_low_accepted)}')


Aceptados del nivel very low: 0


### 7.5 Aplicar todos los fuzzy matches aceptados

In [23]:
all_fuzzy    = pd.concat([auto_matches, review_accepted, low_accepted, very_low_accepted], ignore_index=True)
fuzzy_lookup = dict(zip(all_fuzzy['player_norm'], all_fuzzy['cg_match']))

df_merged['player_norm_fuzzy'] = df_merged.apply(
    lambda r: fuzzy_lookup.get(r['player_norm'], r['player_norm'])
    if pd.isna(r['gross_annual_eur']) else r['player_norm'],
    axis=1
)

df_final = (
    df_merged
    .drop(columns=['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality'])
    .merge(
        df_cg[['player_norm', 'team_norm', 'gross_weekly_eur', 'gross_annual_eur',
               'position', 'age', 'nationality']],
        left_on=['player_norm_fuzzy', 'team_norm'],
        right_on=['player_norm', 'team_norm'],
        how='left'
    )
    .drop(columns=['player_norm_y', 'player_norm_fuzzy'])
    .rename(columns={'player_norm_x': 'player_norm'})
)

matched_final = df_final['gross_annual_eur'].notna().sum()
print(f'Resultado final: {matched_final}/{len(df_final)} ({matched_final/len(df_final):.1%})')
print(f'Sin salario:     {len(df_final) - matched_final}')

Resultado final: 530/580 (91.4%)
Sin salario:     50


## 8. Revisión de jugadores sin salario

Ordenados por equipo y minutos jugados para identificar si alguno debería tener salario.

In [24]:
sin_salario = (
    df_final[df_final['gross_annual_eur'].isna()]
    [['player', 'team', 'minutesPlayed', 'appearances', 'goals', 'assists']]
    .sort_values(['team', 'minutesPlayed'], ascending=[True, False])
    .reset_index(drop=True)
)

pd.set_option('display.max_rows', None)
print(f'Total sin salario: {len(sin_salario)}')
display(sin_salario)
pd.reset_option('display.max_rows')

Total sin salario: 50


,player,team,minutesPlayed,appearances,goals,assists
0,Milad Mohammadi,Adana Demirspor,180,2,0,0
1,Yusuf Bugra Demirkiran,Adana Demirspor,172,3,0,0
2,Édouard Michut,Adana Demirspor,169,2,0,0
3,José Rodríguez,Adana Demirspor,144,2,0,0
4,Ali Arda Yildiz,Adana Demirspor,106,2,0,0
5,Gokdeniz Tunc,Adana Demirspor,61,1,0,0
6,Demir Yavuz,Adana Demirspor,29,1,0,0
7,Sefa Gulay,Adana Demirspor,20,1,0,0
8,Ali Fidan,Adana Demirspor,17,2,0,0
9,Yucel Gurol,Adana Demirspor,12,1,0,0


### 8.1 Comparación manual por equipo

Para cada equipo con jugadores sin salario se muestra la plantilla completa de Capology
ordenada alfabéticamente por nombre normalizado, facilitando la detección visual de matches fallidos.

In [25]:
equipos_sin_salario = sin_salario['team'].unique()

for equipo in sorted(equipos_sin_salario):
    sf_jugadores = sin_salario[sin_salario['team'] == equipo][['player', 'minutesPlayed']].sort_values('player')

    equipo_norm  = normalize(equipo)
    cg_jugadores = (
        df_cg[df_cg['team_norm'] == equipo_norm][['player', 'player_norm']]
        .sort_values('player_norm')
        .reset_index(drop=True)
    )

    print(f'\n{"="*60}')
    print(f'  {equipo}  —  SF sin salario:')
    display(sf_jugadores.reset_index(drop=True))
    print(f'  CG plantilla completa:')
    display(cg_jugadores)


  Adana Demirspor  —  SF sin salario:


,player,minutesPlayed
0,Ahmet Yilmaz,2
1,Ali Arda Yildiz,106
2,Ali Fidan,17
3,Demir Yavuz,29
4,Gokdeniz Tunc,61
5,José Rodríguez,144
6,Milad Mohammadi,180
7,Samet Duyur,8
8,Sefa Gulay,20
9,Yucel Gurol,12


  CG plantilla completa:


,player,player_norm
0,Abat Aymbetov,abat aymbetov
1,Abdulsamet Burak,abdulsamet burak
2,Aksel Aktaş,aksel aktas
3,Ali Yavuz Kol,ali yavuz kol
4,Andreaw Gravillon,andreaw gravillon
5,Arda Kurtulan,arda kurtulan
6,Breyton Fougeu,breyton fougeu
7,Bünyamin Balat,bunyamin balat
8,Burhan Ersoy,burhan ersoy
9,Deniz Dönmezer,deniz donmezer



  Alanyaspor  —  SF sin salario:


...

,player,player_norm
0,Achilleas Poungouras,achilleas poungouras
1,Alaaddin Okumuş,alaaddin okumus
2,Alex Pritchard,alex pritchard
3,Ali Şaşal Vural,ali sasal vural
4,Azizbek Turgunboev,azizbek turgunboev
5,Bekir Turaç Böke,bekir turac boke
6,Bengali-Fodé Koita,bengali fode koita
7,Charilaos Charisis,charilaos charisis
8,Djordje Nikolic,djordje nikolic
9,Efkan Bekiroğlu,efkan bekiroglu



  Trabzonspor  —  SF sin salario:


,player,minutesPlayed
0,Boran Başkan,25
1,Mahmoud Trézéguet,90
2,Taha Emre Ince,11


  CG plantilla completa:


,player,player_norm
0,Ahmet Yıldırım,ahmet yldrm
1,Ali Şahin Yılmaz,ali sahin ylmaz
2,Anthony Nwakaeme,anthony nwakaeme
3,Arif Boşluk,arif bosluk
4,Arseniy Batagov,arseniy batagov
5,Batista Mendy,batista mendy
6,Borna Barisic,borna barisic
7,Cihan Çanak,cihan canak
8,Danylo Sikan,danylo sikan
9,Denis Drăguș,denis dragus


In [28]:
# ── Matches manuales (nombres muy distintos o traspasos invernales) ──
# Formato: (player_norm_sf, team_norm_sf): (player_norm_cg, team_norm_cg)
MANUAL_MATCHES = {
    ('juan santos da silva', 'goztepe')       : ('juan', 'goztepe'),
    ('guilherme haubert sitya', 'konyaspor')  : ('guilherme', 'konyaspor'),
    ('marius mouandilmadji', 'samsunspor')    : ('marius', 'samsunspor'),
}
# ────────────────────────────────────────────────────────────────────
print(f'Matches manuales definidos: {len(MANUAL_MATCHES)}')


Matches manuales definidos: 3


In [29]:
# Aplicar matches manuales sobre los que siguen sin salario
for (p_sf, t_sf), (p_cg, t_cg) in MANUAL_MATCHES.items():
    mask = (df_final['player_norm'] == p_sf) & (df_final['team_norm'] == t_sf) & (df_final['gross_annual_eur'].isna())
    datos_cg = df_cg[(df_cg['player_norm'] == p_cg) & (df_cg['team_norm'] == t_cg)]
    if not datos_cg.empty and mask.any():
        for col in ['gross_weekly_eur', 'gross_annual_eur', 'position', 'age', 'nationality']:
            df_final.loc[mask, col] = datos_cg[col].values[0]
        print(f'✅ Match manual aplicado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')
    else:
        print(f'⚠️  No encontrado: {p_sf} ({t_sf}) → {p_cg} ({t_cg})')

matched_tras_manual = df_final['gross_annual_eur'].notna().sum()
print(f'\nTras matches manuales: {matched_tras_manual}/{len(df_final)} ({matched_tras_manual/len(df_final):.1%})')

✅ Match manual aplicado: juan santos da silva (goztepe) → juan (goztepe)
✅ Match manual aplicado: guilherme haubert sitya (konyaspor) → guilherme (konyaspor)
✅ Match manual aplicado: marius mouandilmadji (samsunspor) → marius (samsunspor)

Tras matches manuales: 533/580 (91.9%)


In [30]:
pd.reset_option('display.max_rows')

## 9. Guardado

Una vez revisado todo, se eliminan las columnas auxiliares y se guarda en `data/master/`.

In [31]:
df_final = df_final.drop(columns=['player_norm', 'team_norm'])

nombre_salida = 'master_turkey_2425.csv'
df_final.to_csv(MASTER_DIR / nombre_salida, index=False)

print(f'✅ Guardado: {nombre_salida}')
print(f'   Jugadores totales:  {len(df_final)}')
print(f'   Con salario:        {df_final["gross_annual_eur"].notna().sum()}')
print(f'   Sin salario (NaN):  {df_final["gross_annual_eur"].isna().sum()}')
print(f'   Columnas:           {df_final.shape[1]}')

✅ Guardado: master_turkey_2425.csv
   Jugadores totales:  580
   Con salario:        533
   Sin salario (NaN):  47
   Columnas:           121
